# Part 4 · Notebook 03 — Historical data, pacing and one schema

**Sessions:** S5 (IB historical data) · S6 (Alpaca historical data & the canonical schema) · [Lesson plan](../../docs/lessons/PART_04_BROKER_CONNECTIVITY.md) · graded labs in [`labs/part04/`](../../labs/part04/)

**You will:**
1. Split a long IB download into requests that walk back in time.
2. Pace requests so IB never bans you (error 162), and see what pacing costs.
3. Normalize IB and Alpaca bars into one canonical, UTC schema.
4. See the two classic data bugs: naive time zones and IEX vs consolidated volume.

How these notebooks work: the setup, data and plotting code is written for you. Cells marked **✍️ Your turn** need a few lines from you.
If your answer does not match yet, the notebook continues with the reference answer so nothing else breaks.
Nothing here connects to a broker: the account rows, bars, ticks and order events are synthetic, shaped like what `ib_async` and `alpaca-py` return.

In [ ]:
import sys
from pathlib import Path
for d in (Path.cwd(), Path.cwd().parent):       # p4lib.py is in notebooks/part04/
    sys.path.insert(0, str(d))
from decimal import Decimal
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import p4lib as p

p.use_course_style()
from datetime import datetime

## 1. Walk back in chunks

An IB historical request is *"`durationStr` of bars ending at `endDateTime`"*, and each bar size has a maximum duration. A long history is therefore many requests, **walking back** from the end. The last chunk is clipped at the start date.

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
from datetime import timedelta

def ib_chunks(start: datetime, end: datetime, chunk_days: int) -> list[tuple[datetime, datetime]]:
    out, cur = [], end
    while cur > start:
        s = ...                                   # ✍️ chunk_days before cur, but never before start
        out.append((s, cur))
        cur = s
    return out

args = (datetime(2025, 1, 1), datetime(2025, 3, 7), 30)
mine = p.attempt(ib_chunks, *args)
mine = p.check("ib_chunks", mine, p.ib_chunks(*args))
[(a.date().isoformat(), b.date().isoformat()) for a, b in mine]

## 2. Pacing: how fast may you ask?

IB's historical-data pacing rules (lesson plan S5): **no more than 6 requests in any 2 seconds**, **no more than 60 in any 10 minutes**, and no identical request within 15 seconds. Break them and you get **error 162** and a temporary ban.

A pacer takes the times you *want* to send requests and returns the times you *may*: a request waits while the last `max_n` sends are all inside the window, until the oldest of them leaves it (`sent[-max_n] + window`).

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
def paced_times(desired: list[float], max_n: int, window: float) -> list[float]:
    sent = []
    for t in desired:
        t = max(t, sent[-1]) if sent else t       # keep the order
        # ✍️ if at least max_n were sent and t is inside the window of the max_n-th last send, wait for it to leave
        ...
        sent.append(t)
    return sent

burst = [0.0] * 20 + [5.0] * 5
mine = paced_times(burst, 6, 2.0)
mine = p.check("paced_times", mine, p.paced_times(burst, 6, 2.0))
mine

What does pacing cost? Say you download two years of 5-minute bars for 10 symbols in 1-week chunks: 1,040 requests, all wanted *now*.

In [ ]:
want = [0.0] * 1040
fast = p.paced_multi(want, [(6, 2.0)])
both = p.paced_multi(want, [(6, 2.0), (60, 600.0)])
fig, ax = plt.subplots()
ax.plot(np.array(fast) / 60, np.arange(1, 1041), label="6 per 2 s only")
ax.plot(np.array(both) / 60, np.arange(1, 1041), label="6 per 2 s and 60 per 10 min")
ax.set(xlabel="minutes after start", ylabel="requests sent", title="Pacing turns a burst into a staircase")
ax.legend(); plt.show()
print(f"6/2s only: {fast[-1] / 60:.1f} min.   With the 10-minute rule: {both[-1] / 3600:.1f} hours.")
print("→ download once into a local store and read from it (notebook 04); never re-download in a backtest loop.")

## 3. Two brokers, one schema

Every bar in our store has the columns `p.CANON` = `ts, symbol, open, high, low, close, volume, source`, with **`ts` a tz-aware UTC timestamp**.

* IB (`util.df(bars)`) gives a `date` column in **exchange local time, without a time zone**.
* Alpaca (`BarSet.df`) gives a MultiIndex `(symbol, timestamp)` already in UTC, with extra columns.

In [ ]:
ib_raw, alp_raw = p.raw_bars()
display(ib_raw.head(3))
display(alp_raw.head(3))

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

For IB: localize the naive times to New York (`p.NY`) and convert to UTC (`.dt.tz_localize(...).dt.tz_convert(...)`).

In [ ]:
def normalize_ib(df: pd.DataFrame, symbol: str) -> pd.DataFrame:
    out = df.rename(columns={"date": "ts"})
    out["ts"] = ...                               # ✍️ naive New York local time → tz-aware UTC
    out["symbol"], out["source"] = symbol, "ib"
    out["volume"] = out["volume"].astype(float)
    return out[p.CANON].sort_values("ts").reset_index(drop=True)

mine = p.attempt(normalize_ib, ib_raw, "SPY")
mine = p.check("normalize_ib", mine, p.normalize_ib(ib_raw, "SPY"))
mine.head()

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

For Alpaca: move the index levels into columns (`reset_index`) and rename `timestamp` to `ts`.

In [ ]:
def normalize_alpaca(df: pd.DataFrame, feed: str = "iex") -> pd.DataFrame:
    out = ...                                     # ✍️ index levels → columns, timestamp → ts
    out["source"] = f"alpaca_{feed}"
    out["volume"] = out["volume"].astype(float)
    return out[p.CANON].sort_values("ts").reset_index(drop=True)

mine = p.attempt(normalize_alpaca, alp_raw)
mine = p.check("normalize_alpaca", mine, p.normalize_alpaca(alp_raw))
mine.head()

## 4. Two classic data bugs

**Bug 1: treating naive exchange time as UTC.** The bars then sit 5 hours early (4 in summer). Worse than not joining at all, some of them *do* join, to the wrong bar.

In [ ]:
ib_ok, alp_ok = p.normalize_ib(ib_raw, "SPY"), p.normalize_alpaca(alp_raw)
ib_bad = ib_ok.assign(ts=ib_raw["date"].dt.tz_localize("UTC"))            # the bug
for name, left in [("naive-as-UTC", ib_bad), ("localized", ib_ok)]:
    joined = left.merge(alp_ok, on="ts", suffixes=("_ib", "_alp"))
    wrong = (joined["close_ib"] != joined["close_alp"]).sum()
    print(f"{name:13s}: {len(joined):3d} of {len(alp_ok)} bars join, {wrong} of them to a different bar")

**Bug 2: mixing feeds.** Alpaca's free IEX feed sees only the trades on IEX, a few percent of consolidated volume. Prices agree, volumes don't. Keep `source` on every bar and never mix feeds inside one strategy.

In [ ]:
share = alp_ok["volume"] / ib_ok["volume"]
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(ib_ok["ts"], ib_ok["close"], label="IB")
axes[0].plot(alp_ok["ts"], alp_ok["close"], "--", label="Alpaca IEX")
axes[0].set(title="Close: the same", ylabel="price"); axes[0].legend(); axes[0].tick_params(axis="x", rotation=30)
axes[1].plot(ib_ok["ts"], share * 100, color=p.PALETTE[1])
axes[1].set(title="Volume: IEX as % of consolidated", ylabel="%"); axes[1].tick_params(axis="x", rotation=30)
plt.tight_layout(); plt.show()
print(f"IEX sees on average {share.mean():.1%} of the volume")

## Wrap-up

* Long histories are many requests walking back in time; pacing makes them slow, so download once and store.
* One schema, UTC everywhere, `source` on every bar.
* Graded version: `labs/part04/week14_data` (`ib_chunks`, `PacingGuard` with all three rules, `normalize_ib`, `normalize_alpaca`).